# Gold — modelo dimensional (estrela) + SCD2 + checkpoint

Issue #38. Constrói o modelo dimensional a partir da Silver, servindo
diretamente os KPIs do dashboard (CARD-006):

- **Dimensões**: `dim_date` (estática) e `dim_customer`, `dim_seller`,
  `dim_product` com **SCD Tipo 2** (versionamento histórico via `MERGE` Delta,
  surrogate key determinística `xxhash64(nk, _hash)`, `valid_from/valid_to/is_current`).
- **Fatos**: `fact_orders` (grão pedido), `fact_order_items` (grão item) e
  `fact_payments` (grão pagamento) com **carga incremental por checkpoint**:
  um watermark por fato (`gold/_control/fact_checkpoints`) garante que cada
  execução processa só o que chegou desde a última.

Ao final, calcula os 4 KPIs + 2 gráficos para validar a camada.

In [ ]:
import os

run_date = "2026-06-17"

minio_endpoint = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
minio_access_key = os.environ.get("MINIO_ROOT_USER", "minioadmin")
minio_secret_key = os.environ.get("MINIO_ROOT_PASSWORD", "minioadmin")
bucket = os.environ.get("DATALAKE_BUCKET", "datalake")

In [ ]:
APP_NAME = "engdb-gold-olist"
EXTRA_PACKAGES = ["org.apache.hadoop:hadoop-aws:3.3.4"]

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder.appName(APP_NAME)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", minio_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)
spark = configure_spark_with_delta_pip(builder, extra_packages=EXTRA_PACKAGES).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

GOLD = f"s3a://{bucket}/gold/olist"
CTRL = f"s3a://{bucket}/gold/_control/fact_checkpoints"

def read_silver(t):
    return spark.read.format("delta").load(f"s3a://{bucket}/silver/olist/{t}")

# ---------- SCD Tipo 2 ----------
def build_scd2_df(df, nk, attrs, sk):
    """Monta o DataFrame de entrada da dimensão com hash de versão e SK determinístico."""
    h = F.sha2(F.concat_ws("||", *[F.coalesce(F.col(a).cast("string"), F.lit("∅")) for a in attrs]), 256)
    return (df.select(nk, *attrs)
              .withColumn("_hash", h)
              .withColumn(sk, F.xxhash64(F.col(nk), F.col("_hash")))
              .withColumn("valid_from", F.to_date(F.lit(run_date)))
              .withColumn("valid_to", F.lit(None).cast("date"))
              .withColumn("is_current", F.lit(True)))

def scd2_upsert(path, incoming, nk, sk):
    if not DeltaTable.isDeltaTable(spark, path):
        incoming.write.format("delta").mode("overwrite").save(path)
        return
    dt = DeltaTable.forPath(spark, path)
    # 1) insere versões novas (SK inédito) como vigentes
    (dt.alias("t").merge(incoming.alias("s"), f"t.{sk} = s.{sk}")
        .whenNotMatchedInsertAll().execute())
    # 2) expira a versão anterior cujo SK deixou de ser o vigente para a mesma NK
    (dt.alias("t").merge(incoming.select(nk, sk).alias("s"),
        f"t.{nk} = s.{nk} AND t.is_current = true AND t.{sk} <> s.{sk}")
        .whenMatchedUpdate(set={"is_current": F.lit(False), "valid_to": F.to_date(F.lit(run_date))})
        .execute())

# ---------- checkpoint (watermark por fato) ----------
def get_watermark(fact_name):
    if not DeltaTable.isDeltaTable(spark, CTRL):
        return None
    r = (spark.read.format("delta").load(CTRL)
            .where(F.col("fact_name") == fact_name).select("watermark").collect())
    return r[0][0] if r else None

def set_watermark(fact_name, wm):
    df = spark.createDataFrame([(fact_name, wm)], "fact_name string, watermark timestamp")
    if not DeltaTable.isDeltaTable(spark, CTRL):
        df.write.format("delta").mode("overwrite").save(CTRL)
    else:
        dt = DeltaTable.forPath(spark, CTRL)
        (dt.alias("t").merge(df.alias("s"), "t.fact_name = s.fact_name")
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

def upsert_fact(path, df, pk):
    if not DeltaTable.isDeltaTable(spark, path):
        df.write.format("delta").mode("overwrite").save(path)
    else:
        dt = DeltaTable.forPath(spark, path)
        cond = " AND ".join([f"t.{k} = s.{k}" for k in pk])
        (dt.alias("t").merge(df.alias("s"), cond)
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

def date_sk(col):
    return F.date_format(col, "yyyyMMdd").cast("int")

In [ ]:
# ===================== DIMENSÕES =====================
orders_s = read_silver("orders")

# dim_date: cobre a faixa de datas dos pedidos
b = orders_s.select(
    F.min("order_purchase_timestamp").alias("dmin"),
    F.max(F.coalesce("order_delivered_customer_date",
                     "order_estimated_delivery_date",
                     "order_purchase_timestamp")).alias("dmax")).collect()[0]
start, end = b["dmin"].date(), b["dmax"].date()
days = (end - start).days + 1
dim_date = (spark.range(0, days)
    .withColumn("full_date", F.expr(f"date_add(to_date('{start}'), cast(id as int))"))
    .withColumn("date_sk", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMM"))
    .withColumn("year_month", F.date_format("full_date", "yyyy-MM"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))
    .drop("id"))
dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{GOLD}/dim_date")
print("[gold] dim_date     linhas =", dim_date.count())

# dim_customer (SCD2)
scd2_upsert(f"{GOLD}/dim_customer",
    build_scd2_df(read_silver("customers"), "customer_id",
        ["customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"],
        "customer_sk"),
    "customer_id", "customer_sk")
print("[gold] dim_customer linhas =", spark.read.format("delta").load(f"{GOLD}/dim_customer").count())

# dim_seller (SCD2)
scd2_upsert(f"{GOLD}/dim_seller",
    build_scd2_df(read_silver("sellers"), "seller_id",
        ["seller_zip_code_prefix", "seller_city", "seller_state"], "seller_sk"),
    "seller_id", "seller_sk")
print("[gold] dim_seller   linhas =", spark.read.format("delta").load(f"{GOLD}/dim_seller").count())

# dim_product (SCD2, com categoria resolvida)
prod = (read_silver("products")
        .join(read_silver("categories").select("category_id", "product_category_name"), "category_id", "left")
        .withColumn("product_category_name", F.coalesce("product_category_name", F.lit("desconhecida"))))
scd2_upsert(f"{GOLD}/dim_product",
    build_scd2_df(prod, "product_id",
        ["product_category_name", "product_weight_g", "product_photos_qty",
         "product_length_cm", "product_height_cm", "product_width_cm"], "product_sk"),
    "product_id", "product_sk")
print("[gold] dim_product  linhas =", spark.read.format("delta").load(f"{GOLD}/dim_product").count())

In [ ]:
# ===================== FATOS (incremental por checkpoint) =====================
orders_s = read_silver("orders")
payments_s = read_silver("payments")
items_s = read_silver("order_items")

def dim_current(name, nk, sk):
    return spark.read.format("delta").load(f"{GOLD}/{name}").where("is_current = true").select(nk, sk)

cust_cur = dim_current("dim_customer", "customer_id", "customer_sk")
prod_cur = dim_current("dim_product", "product_id", "product_sk")
sell_cur = dim_current("dim_seller", "seller_id", "seller_sk")

pay_by_order = payments_s.groupBy("order_id").agg(
    F.sum("payment_value").alias("order_payment_value"),
    F.count(F.lit(1)).alias("payment_count"))

# ---- fact_orders (grão: pedido) ----
wm = get_watermark("fact_orders")
base = orders_s if wm is None else orders_s.where(F.col("order_purchase_timestamp") > F.lit(wm))
n = base.count()
print(f"[gold] fact_orders      watermark={wm} -> {n} novos pedidos")
if n > 0:
    fo = (base.join(cust_cur, "customer_id", "left")
          .join(pay_by_order, "order_id", "left")
          .withColumn("date_sk", date_sk("order_purchase_timestamp"))
          .withColumn("delivery_days", F.datediff("order_delivered_customer_date", "order_approved_at"))
          .withColumn("is_delivered", F.col("order_status") == "delivered")
          .withColumn("is_canceled", F.col("order_status") == "canceled")
          .select("order_id", "customer_sk", "date_sk", "order_status",
                  "order_purchase_timestamp",
                  F.coalesce("order_payment_value", F.lit(0).cast("decimal(12,2)")).alias("order_payment_value"),
                  F.coalesce("payment_count", F.lit(0)).alias("payment_count"),
                  "delivery_days", "is_delivered", "is_canceled"))
    upsert_fact(f"{GOLD}/fact_orders", fo, ["order_id"])
    set_watermark("fact_orders", base.agg(F.max("order_purchase_timestamp")).collect()[0][0])
print("[gold] fact_orders      total =", spark.read.format("delta").load(f"{GOLD}/fact_orders").count())

# ---- fact_order_items (grão: item) ----
order_ts = orders_s.select("order_id", "order_purchase_timestamp")
oi = items_s.join(order_ts, "order_id", "inner")
wm = get_watermark("fact_order_items")
base = oi if wm is None else oi.where(F.col("order_purchase_timestamp") > F.lit(wm))
n = base.count()
print(f"[gold] fact_order_items watermark={wm} -> {n} novos itens")
if n > 0:
    foi = (base.join(prod_cur, "product_id", "left")
           .join(sell_cur, "seller_id", "left")
           .withColumn("date_sk", date_sk("order_purchase_timestamp"))
           .select("order_item_id", "order_id", "product_sk", "seller_sk", "date_sk",
                   "price", "freight_value", "order_purchase_timestamp"))
    upsert_fact(f"{GOLD}/fact_order_items", foi, ["order_item_id"])
    set_watermark("fact_order_items", base.agg(F.max("order_purchase_timestamp")).collect()[0][0])
print("[gold] fact_order_items total =", spark.read.format("delta").load(f"{GOLD}/fact_order_items").count())

# ---- fact_payments (grão: pagamento) ----
order_ref = orders_s.select("order_id", "customer_id", "order_purchase_timestamp")
pay = payments_s.join(order_ref, "order_id", "inner")
wm = get_watermark("fact_payments")
base = pay if wm is None else pay.where(F.col("order_purchase_timestamp") > F.lit(wm))
n = base.count()
print(f"[gold] fact_payments    watermark={wm} -> {n} novos pagamentos")
if n > 0:
    fp = (base.join(cust_cur, "customer_id", "left")
          .withColumn("date_sk", date_sk("order_purchase_timestamp"))
          .select("payment_id", "order_id", "customer_sk", "date_sk", "payment_type",
                  "payment_sequential", "payment_installments", "payment_value",
                  "order_purchase_timestamp"))
    upsert_fact(f"{GOLD}/fact_payments", fp, ["payment_id"])
    set_watermark("fact_payments", base.agg(F.max("order_purchase_timestamp")).collect()[0][0])
print("[gold] fact_payments    total =", spark.read.format("delta").load(f"{GOLD}/fact_payments").count())

In [ ]:
# ===================== VALIDAÇÃO: KPIs do dashboard a partir da Gold =====================
fo = spark.read.format("delta").load(f"{GOLD}/fact_orders")
foi = spark.read.format("delta").load(f"{GOLD}/fact_order_items")
dp = spark.read.format("delta").load(f"{GOLD}/dim_product").where("is_current = true")
dd = spark.read.format("delta").load(f"{GOLD}/dim_date")

receita = fo.agg(F.sum("order_payment_value")).collect()[0][0] or 0
qtd = fo.select("order_id").distinct().count()
tme = fo.where("is_delivered").agg(F.avg("delivery_days")).collect()[0][0] or 0

print("================ KPIs (camada Gold) ================")
print(f"KPI1 Receita Total          : R$ {receita:,.2f}")
print(f"KPI2 Quantidade de Pedidos  : {qtd:,}")
print(f"KPI3 Ticket Médio           : R$ {(receita / qtd if qtd else 0):,.2f}")
print(f"KPI4 Tempo Médio de Entrega : {tme:.1f} dias")

print("\n-- Gráfico 1: Vendas por mês (primeiros 6) --")
(fo.join(dd, "date_sk").groupBy("year_month")
   .agg(F.round(F.sum("order_payment_value"), 2).alias("receita"))
   .orderBy("year_month").show(6, False))

print("-- Gráfico 2: Top 10 categorias (qtd vendida) --")
(foi.join(dp, "product_sk").groupBy("product_category_name")
    .agg(F.count(F.lit(1)).alias("qtd_vendida"))
    .orderBy(F.desc("qtd_vendida")).show(10, False))

spark.stop()